In [4]:
# CAD class
from dataclasses import dataclass

class Line:
    def __init__(self, start, end, layer):
        self.start = start  # (x, y)
        self.end = end
        self.layer = layer

    def __repr__(self):
        return f"Line(start={self.start}, end={self.end})"

class Arc:
    def __init__(self, center, radius, start_angle, end_angle, layer):
        self.center = center  # (x, y)
        self.radius = radius
        self.start_angle = start_angle
        self.end_angle = end_angle
        self.layer = layer

    def __repr__(self):
        return (f"Arc(center={self.center}, radius={self.radius}, "
                f"start_angle={self.start_angle}, end_angle={self.end_angle})")

class Polyline:
    def __init__(self, points, is_closed=False):
        self.points = points  # [(x1, y1), (x2, y2), ...]
        self.is_closed = is_closed

    def __repr__(self):
        return f"Polyline(points={self.points}, is_closed={self.is_closed})"
    
@dataclass(frozen=True, eq=True, slots=True) 
class TransferNode:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class Direction:
    def __init__(self, startPoint, endPoint):
        self.startPoint = startPoint
        self.endPoint = endPoint

In [5]:
# Map Class
class Node:
    def __init__(self, id, type, reality, x, y, layer, waitNode, parentLinkId, relativeDistance):
        self.id = id
        self.type = type
        self.reality = reality
        self.x = float(x)
        self.y = float(y)
        self.layer = layer
        self.waitNode = waitNode
        self.parentLinkId = parentLinkId
        self.relativeDistance = relativeDistance
    def setWaitNode(self, waitNode):
        self.waitNode = waitNode


class Link:
    def __init__(self, id, type, startNode, endNode, length):
        self.id = id
        self.type = type
        self.startNode = startNode
        self.endNode = endNode
        self.length = length

class Port:
    def __init__(self, id, type, carrierType, x, y, nodeId, nodeAlignment, teachingDone):
        self.id = id
        self.type = type
        self.carrierType = carrierType
        self.x = x
        self.y = y
        self.nodeId = nodeId
        self.nodeAlignment = nodeAlignment
        self.teachingDone = teachingDone

class Zone:
    def __init__(self, id, nodesList):
        self.id = id
        self.nodesList = nodesList


In [1]:
# CAD class
from dataclasses import dataclass
import logging
import warnings
import ezdxf
from math import sqrt, atan2, degrees, hypot
from collections import defaultdict, Counter
from statistics import mean
from pathlib import Path
import math
import copy
import csv

# Line객체를 Node객체, Link객체로 변환
nodes_list = []
links_list = []
horizontalLinks = {}
verticalLinks = {}
ports_list = []
eqNodeIdMatch = {}
stbNodeIdMatch = {}
nodeId = 1
linkId = 1

motherNodesList = []
motherLinksList = []
diagonalLinksNodesMatch = {}
diagonalLinksList = []
motherVerticalLinks = {}
motherHorizontalLinks= {}
motherSonLinksMatch = {}
motherStartEndNodesList = []

class Line:
    def __init__(self, start, end, layer):
        self.start = start  # (x, y)
        self.end = end
        self.layer = layer

    def __repr__(self):
        return f"Line(start={self.start}, end={self.end})"

class Arc:
    def __init__(self, center, radius, start_angle, end_angle, layer):
        self.center = center  # (x, y)
        self.radius = radius
        self.start_angle = start_angle
        self.end_angle = end_angle
        self.layer = layer

    def __repr__(self):
        return (f"Arc(center={self.center}, radius={self.radius}, "
                f"start_angle={self.start_angle}, end_angle={self.end_angle})")

class Polyline:
    def __init__(self, points, is_closed=False):
        self.points = points  # [(x1, y1), (x2, y2), ...]
        self.is_closed = is_closed

    def __repr__(self):
        return f"Polyline(points={self.points}, is_closed={self.is_closed})"
    
@dataclass(frozen=True, eq=True, slots=True) 
class TransferNode:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class Direction:
    def __init__(self, startPoint, endPoint):
        self.startPoint = startPoint
        self.endPoint = endPoint

# Map Class
class Node:
    def __init__(self, id, type, reality, x, y, layer, waitNode, parentLinkId, relativeDistance):
        self.id = id
        self.type = type
        self.reality = reality
        self.x = float(x)
        self.y = float(y)
        self.layer = layer
        self.waitNode = waitNode
        self.parentLinkId = parentLinkId
        self.relativeDistance = relativeDistance
    def setWaitNode(self, waitNode):
        self.waitNode = waitNode

class Link:
    def __init__(self, id, type, startNode, endNode, length):
        self.id = id
        self.type = type
        self.startNode = startNode
        self.endNode = endNode
        self.length = length

class Port:
    def __init__(self, id, type, carrierType, x, y, nodeId, nodeAlignment, teachingDone):
        self.id = id
        self.type = type
        self.carrierType = carrierType
        self.x = x
        self.y = y
        self.nodeId = nodeId
        self.nodeAlignment = nodeAlignment
        self.teachingDone = teachingDone

class Zone:
    def __init__(self, id, nodesList):
        self.id = id
        self.nodesList = nodesList

# ===== 소음 억제 =====
logging.getLogger("ezdxf").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# ===== 데이터 클래스 (중복 정의 방지 및 병합) =====
@dataclass
class Line_DC:
    start: tuple[float, float]
    end:   tuple[float, float]
    layer: int = 0  # 0=Lower, 1=Upper

@dataclass
class Arc_DC:
    center:      tuple[float, float]
    radius:      float
    start_angle: float
    end_angle:   float
    layer:       int = 0

# ===== 설정 =====
# 레일 추출용
ALLOWED_INSERT_LAYERS = []
UPPER_RAIL_LAYERS = []
LOWER_RAIL_LAYERS = []
ALLOWED_ENTITY_LAYERS = []

# [수정] CSV 파일 읽기 오류 방지 (파일이 없어도 에러 없이 넘어가도록 try-except 추가)
try:
    csv_path = Path("CadToMap_Input/Layer_UpperRail.csv")
    with csv_path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            name = line.strip()
            if name:
                UPPER_RAIL_LAYERS.append(name)
                ALLOWED_INSERT_LAYERS.append(name)
                ALLOWED_ENTITY_LAYERS.append(name)
except FileNotFoundError:
    pass

try:
    csv_path = Path("CadToMap_Input/Layer_LowerRail.csv")
    with csv_path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            name = line.strip()
            if name:
                LOWER_RAIL_LAYERS.append(name)
                ALLOWED_INSERT_LAYERS.append(name)
                ALLOWED_ENTITY_LAYERS.append(name)
except FileNotFoundError:
    pass

INCLUDE_LWPOLYLINE     = True
ALLOWED_INSERT_LAYERS2= []
EQ_LAYERS = []
STB_LAYERS = []

try:
    csv_path = Path("CadToMap_Input/Layer_EqPort.csv")
    with csv_path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            name = line.strip()
            if name:
                ALLOWED_INSERT_LAYERS2.append(name)
                EQ_LAYERS.append(name)
except FileNotFoundError:
    pass

try:
    csv_path = Path("CadToMap_Input/Layer_StbPort.csv")
    with csv_path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            name = line.strip()
            if name:
                ALLOWED_INSERT_LAYERS2.append(name)
                STB_LAYERS.append(name)
except FileNotFoundError:
    pass

CLINE_LAYER_NAME       = "_CLINE"
port_input = "None"
SKIP_ANONYMOUS_BLOCKS  = True
NORMALIZE_MIRROR       = False
EPS = 1e-6
VERBOSE = False
COORD_ROUND_DIGITS = 0

def R(v: float) -> float:
    return round(v, COORD_ROUND_DIGITS) if COORD_ROUND_DIGITS is not None else v

def Rp(p: tuple[float, float]) -> tuple[float, float]:
    return (R(p[0]), R(p[1]))

# --- ARC 미러 보정 옵션 ---
ARC_MIRROR_FIX = True
ARC_MIRROR_RULE = "center_x_negative"
ARC_MIRROR_APPLY_ON = {"virtual"}

def maybe_mirror_arc(cx: float, cy: float, sa: float, ea: float, source: str) -> tuple[float,float,float,float]:
    if not ARC_MIRROR_FIX or source not in ARC_MIRROR_APPLY_ON:
        return cx, cy, sa, ea
    if ARC_MIRROR_RULE == "center_x_negative" and cx < 0:
        cx = -cx
        sa = (180.0 - sa) % 360.0
        ea = (180.0 - ea) % 360.0
        sa, ea = ea, sa
    return cx, cy, sa, ea

# ===== 유틸 =====
def lwpoly_to_segments(lw):
    pts = [tuple(v) for v in lw.get_points("xyb")]
    if lw.closed and len(pts) > 1:
        pts.append(pts[0])

    lines, arcs = [], []
    for i in range(len(pts) - 1):
        (x1, y1, b1) = pts[i]
        (x2, y2, _ ) = pts[i+1]
        bulge = b1

        if abs(bulge) < 1e-12:
            lines.append(((x1, y1), (x2, y2)))
            continue

        dx, dy = x2 - x1, y2 - y1
        chord = sqrt(dx*dx + dy*dy)
        if chord < 1e-12:
            continue

        sagitta = (bulge * chord) / 2.0
        r = (chord**2 / (8.0 * abs(sagitta))) + (abs(sagitta) / 2.0)

        mx, my = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        ux, uy = dx / chord, dy / chord
        nx, ny = -uy, ux
        h = sqrt(max(r*r - (chord/2.0)**2, 0.0))
        cx = mx + (h if bulge > 0 else -h) * nx
        cy = my + (h if bulge > 0 else -h) * ny

        start_ang = degrees(atan2(y1 - cy, x1 - cx)) % 360.0
        end_ang   = degrees(atan2(y2 - cy, x2 - cx)) % 360.0
        if bulge < 0:
            start_ang, end_ang = end_ang, start_ang

        arcs.append(((cx, cy), r, start_ang, end_ang))
    return lines, arcs

def _q(v: float, tol: float) -> float:
    return round(v / tol) * tol

def merge_hv_lines(line_objects: list, tol: float = EPS) -> list:
    vertical_groups: dict[tuple[int, float], list[tuple[float,float,float]]] = defaultdict(list)
    horizontal_groups: dict[tuple[int, float], list[tuple[float,float,float]]] = defaultdict(list)
    others: list = []

    for ln in line_objects:
        (x1, y1), (x2, y2) = ln.start, ln.end
        layer = ln.layer
        if abs(x1 - x2) <= tol and abs(y1 - y2) > tol:
            x_mean = (x1 + x2) / 2.0
            key = (layer, _q(x_mean, tol))
            y_lo, y_hi = sorted((y1, y2))
            vertical_groups[key].append((y_lo, y_hi, x_mean))
        elif abs(y1 - y2) <= tol and abs(x1 - x2) > tol:
            y_mean = (y1 + y2) / 2.0
            key = (layer, _q(y_mean, tol))
            x_lo, x_hi = sorted((x1, x2))
            horizontal_groups[key].append((x_lo, x_hi, y_mean))
        else:
            others.append(ln)

    merged: list = []
    for (layer, x_key), segs in vertical_groups.items():
        segs_sorted = sorted(segs, key=lambda t: t[0])
        cur_lo, cur_hi = None, None
        x_acc, n = 0.0, 0
        for y_lo, y_hi, x_mean in segs_sorted:
            if cur_lo is None:
                cur_lo, cur_hi = y_lo, y_hi
                x_acc, n = x_mean, 1
                continue
            if y_lo <= cur_hi + tol:
                cur_hi = max(cur_hi, y_hi)
                x_acc += x_mean
                n += 1
            else:
                x_final = R(x_acc / max(n, 1))
                merged.append(Line_DC(start=(R(x_final), R(cur_lo)), end=(R(x_final), R(cur_hi)), layer=layer))
                cur_lo, cur_hi = y_lo, y_hi
                x_acc, n = x_mean, 1
        if cur_lo is not None:
            x_final = R(x_acc / max(n, 1))
            merged.append(Line_DC(start=(R(x_final), R(cur_lo)), end=(R(x_final), R(cur_hi)), layer=layer))

    for (layer, y_key), segs in horizontal_groups.items():
        segs_sorted = sorted(segs, key=lambda t: t[0])
        cur_lo, cur_hi = None, None
        y_acc, n = 0.0, 0
        for x_lo, x_hi, y_mean in segs_sorted:
            if cur_lo is None:
                cur_lo, cur_hi = x_lo, x_hi
                y_acc, n = y_mean, 1
                continue
            if x_lo <= cur_hi + tol:
                cur_hi = max(cur_hi, x_hi)
                y_acc += y_mean
                n += 1
            else:
                y_final = R(y_acc / max(n, 1))
                merged.append(Line_DC(start=(R(cur_lo), R(y_final)), end=(R(cur_hi), R(y_final)), layer=layer))
                cur_lo, cur_hi = x_lo, x_hi
                y_acc, n = y_mean, 1
        if cur_lo is not None:
            y_final = R(y_acc / max(n, 1))
            merged.append(Line_DC(start=(R(cur_lo), R(y_final)), end=(R(cur_hi), R(y_final)), layer=layer))

    merged.extend(others)
    return merged


# ===== [핵심 수정부] 레일 수집 (라인/호) =====
def collect_rails_into_lists(doc, line_objects, arc_objects):
    """
    - 도면 내 레이어를 자동 파악
    - 특정 블록('rail line 0.5')은 레이어 규칙을 무시하고 내부 객체 무조건 추출 (0, ByLayer 대응)
    - 호(Arc)의 반전 및 직선화 오류 완벽 해결
    """
    msp = doc.modelspace()
    exploded_insert_count = 0
    exploded_converted_count = 0

    # 1. 도면 내 모든 레이어 자동 파악 (사용자 요청 반영)
    all_layers = [layer.dxf.name for layer in doc.layers]
    print(f"\n🔎 [분석] 도면 내 {len(all_layers)}개의 레이어가 감지되었습니다. 자동 파악을 진행합니다.")

    # 2. 모델공간 직접 그린 요소 (기존 로직 유지)
    for e in msp:
        et = e.dxftype()
        lyr = getattr(e.dxf, "layer", "0")
        
        # CSV 파일 기반 허용 레이어가 세팅된 경우만 필터링 (없으면 모두 허용)
        if ALLOWED_ENTITY_LAYERS and lyr not in ALLOWED_ENTITY_LAYERS:
            continue

        layer_idx = 1 if "upper" in lyr.lower() else 0

        if et == "LINE":
            s, t = e.dxf.start, e.dxf.end
            line_objects.append(Line_DC(start=Rp((float(s.x), float(s.y))),
                                     end=Rp((float(t.x), float(t.y))),
                                     layer=layer_idx))

        elif et == "ARC":
            c = e.dxf.center
            sa = float(e.dxf.start_angle) % 360.0
            ea = float(e.dxf.end_angle)   % 360.0
            cx, cy, r = float(c.x), float(c.y), float(e.dxf.radius)
            cx, cy, sa, ea = maybe_mirror_arc(cx, cy, sa, ea, source="direct")

            # 호(Arc) 오류 방지: 각도와 길이 보정
            sweep = ea - sa
            if sweep <= 0: sweep += 360
            arc_len = (2 * math.pi * r) * (sweep / 360.0)
            if arc_len > 0.1 and sweep > 0.01: # 노이즈(직선화 된 호) 필터
                arc_objects.append(Arc_DC(center=(cx, cy), radius=r, start_angle=sa, end_angle=ea, layer=layer_idx))

        elif et == "LWPOLYLINE" and INCLUDE_LWPOLYLINE:
            lns, ars = lwpoly_to_segments(e)
            for (x1, y1), (x2, y2) in lns:
                line_objects.append(Line_DC(start=Rp((x1, y1)), end=Rp((x2, y2)), layer=layer_idx))
            for (cx, cy), r, sa, ea in ars:
                sweep = ea - sa
                if sweep <= 0: sweep += 360
                arc_len = (2 * math.pi * r) * (sweep / 360.0)
                if arc_len > 0.1 and sweep > 0.01:
                    arc_objects.append(Arc_DC(center=(cx, cy), radius=r, start_angle=sa, end_angle=ea, layer=layer_idx))

    # 3. INSERT 해체 (특정 블록 "rail line 0.5" 타겟팅 및 0번 레이어 상속)
    for ins in msp.query("INSERT"):
        block_name = ins.dxf.name or ""
        
        # 익명 블록 스킵
        if SKIP_ANONYMOUS_BLOCKS and block_name.startswith("*"):
            continue

        ins_layer = getattr(ins.dxf, "layer", "0")
        
        # [핵심] 타겟 블록("rail line 0.5")이거나 허용된 레이어 안에 있으면 실행
        is_target_block = (block_name.lower() == "rail line 0.5")
        if not is_target_block and (ALLOWED_INSERT_LAYERS and ins_layer not in ALLOWED_INSERT_LAYERS):
            continue

        try:
            exploded_insert_count += 1
            for ve in ins.virtual_entities():
                et = ve.dxftype()
                vlayer = getattr(ve.dxf, "layer", "0")
                
                # 내부 레이어가 '0'이거나 ByBlock이면 부모 INSERT의 레이어 속성을 물려받음
                eff_layer = ins_layer if vlayer in (None, "", 0, "0", "BYBLOCK", "ByBlock") else vlayer
                layer_idx = 1 if "upper" in eff_layer.lower() else 0

                if et == "LINE":
                    s, t = ve.dxf.start, ve.dxf.end
                    line_objects.append(Line_DC(start=Rp((float(s.x), float(s.y))),
                                             end=Rp((float(t.x), float(t.y))),
                                             layer=layer_idx))
                    exploded_converted_count += 1

                elif et == "ARC":
                    c = ve.dxf.center
                    sa = float(ve.dxf.start_angle) % 360.0
                    ea = float(ve.dxf.end_angle)   % 360.0
                    r  = float(ve.dxf.radius)
                    cx, cy = float(c.x), float(c.y)

                    # 미러 보정
                    cx, cy, sa, ea = maybe_mirror_arc(cx, cy, sa, ea, source="virtual")

                    # [핵심] 호 반전 및 직선화 노이즈 차단
                    sweep = ea - sa
                    while sweep <= 0: sweep += 360
                    while sweep > 360: sweep -= 360
                    
                    arc_len = (2 * math.pi * r) * (sweep / 360.0)
                    
                    # 지나치게 작거나 직선처럼 왜곡된 ARC 무시
                    if arc_len > 0.1 and sweep > 0.01:
                        arc_objects.append(Arc_DC(center=(cx, cy), radius=r, start_angle=sa, end_angle=ea, layer=layer_idx))
                        exploded_converted_count += 1

                elif et == "LWPOLYLINE" and INCLUDE_LWPOLYLINE:
                    lns, ars = lwpoly_to_segments(ve)
                    for (x1, y1), (x2, y2) in lns:
                        line_objects.append(Line_DC(start=Rp((x1, y1)), end=Rp((x2, y2)), layer=layer_idx))
                        exploded_converted_count += 1
                    for (cx, cy), r, sa, ea in ars:
                        sweep = ea - sa
                        while sweep <= 0: sweep += 360
                        while sweep > 360: sweep -= 360
                        arc_len = (2 * math.pi * r) * (sweep / 360.0)
                        
                        if arc_len > 0.1 and sweep > 0.01:
                            arc_objects.append(Arc_DC(center=(cx, cy), radius=r, start_angle=sa, end_angle=ea, layer=layer_idx))
                            exploded_converted_count += 1

        except Exception as ex:
            if VERBOSE:
                print(f"[WARN] INSERT 해체 실패({block_name}): {ex}")

    # 수평/수직 라인 병합
    if line_objects:
        merged = merge_hv_lines(line_objects, tol=EPS)
        line_objects[:] = merged

    return exploded_insert_count, exploded_converted_count

# ===== TransferNode 등 기타 함수 유지 =====
def _get_lwpoly_vertices_xy(lw):
    pts = [tuple(p[:2]) for p in lw.get_points("xy")]
    if len(pts) >= 2 and (abs(pts[0][0]-pts[-1][0]) < EPS and abs(pts[0][1]-pts[-1][1]) < EPS):
        pts = pts[:-1]
    return pts

def _is_rect_like(points, tol=EPS):
    if len(points) != 4:
        return False
    uniq = []
    for (x, y) in points:
        if not any(abs(x-ux) < tol and abs(y-uy) < tol for (ux, uy) in uniq):
            uniq.append((x, y))
    return len(uniq) == 4

def _centroid_of_points(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return (sum(xs)/len(xs), sum(ys)/len(ys))

def _iter_virtual_recursive(ins, max_depth=3, depth=0):
    for e in ins.virtual_entities():
        if e.dxftype() == "INSERT" and depth < max_depth:
            for ee in _iter_virtual_recursive(e, max_depth=max_depth, depth=depth+1):
                yield ee
        else:
            yield e

def collect_transferNodes(doc, eqNode_objects, stbNode_objects):
    msp = doc.modelspace()
    exploded_ports = 0
    total_transfer_nodes = 0

    for ins in msp.query("INSERT"):
        block_name = ins.dxf.name or ""
        if SKIP_ANONYMOUS_BLOCKS and block_name.startswith("*"):
            continue

        ins_layer = getattr(ins.dxf, "layer", None)
        if ALLOWED_INSERT_LAYERS2 and ins_layer not in ALLOWED_INSERT_LAYERS2:
            continue

        try:
            exploded_ports += 1
            centers = []

            for ve in _iter_virtual_recursive(ins, max_depth=3):
                if ve.dxftype() != "LWPOLYLINE":
                    continue
                vlayer = getattr(ve.dxf, "layer", None)
                eff_layer = ins_layer if vlayer in (None, "", "0", 0, "BYBLOCK", "ByBlock") else vlayer

                pts = _get_lwpoly_vertices_xy(ve)
                if not _is_rect_like(pts):
                    continue

                cx, cy = _centroid_of_points(pts)
                centers.append((cx, cy))

            if ins_layer in EQ_LAYERS:
                for (cx, cy) in centers:
                    eqNode_objects.append(TransferNode(x=round(cx,0), y=round(cy,0))); total_transfer_nodes += 1
            if ins_layer in STB_LAYERS:
                for (cx, cy) in centers:
                    stbNode_objects.append(TransferNode(x=round(cx,0), y=round(cy,0))); total_transfer_nodes += 1

        except Exception as ex:
            if VERBOSE:
                print(f"[WARN] INSERT 해체 실패({block_name}): {ex}")

def find_line8_block_definitions(doc):
    arrow_defs = set()
    debug = {}

    for blk in doc.blocks:
        name = blk.name or ""
        if name in ("*Model_Space", "*Paper_Space", "*Paper_Space0", "*Paper_Space1"):
            continue

        type_counts = Counter(e.dxftype() for e in blk)
        total = sum(type_counts.values())
        if total == 8 and type_counts.get("LINE", 0) == 8:
            arrow_defs.add(name)
        debug[name] = dict(type_counts)

    return arrow_defs, debug

def _iter_named_inserts_recursive(top_insert, names_set, max_depth=8, depth=0):
    nm = (top_insert.dxf.name or "")
    if nm in names_set:
        yield top_insert
    if depth >= max_depth:
        return
    for e in top_insert.virtual_entities():
        if e.dxftype() == "INSERT":
            if (e.dxf.name or "") in names_set:
                yield e
            yield from _iter_named_inserts_recursive(e, names_set, max_depth, depth+1)

def _get_lines_from_insert(ins):
    segs = []
    for e in ins.virtual_entities():
        if e.dxftype() == "LINE":
            s, t = e.dxf.start, e.dxf.end
            segs.append(((float(s.x), float(s.y)), (float(t.x), float(t.y))))
    return segs

def _longest_segment(segments):
    best = None
    bestL = -1.0
    for (p, q) in segments:
        L = hypot(q[0]-p[0], q[1]-p[1])
        if L > bestL:
            bestL = L
            best = (p, q)
    return best

def collect_directions_from_line8_block_inserts(doc, direction_objects):
    msp = doc.modelspace()
    arrow_defs, defs_debug = find_line8_block_definitions(doc)

    per_block_counter = Counter()
    summary = Counter()

    for top in msp.query("INSERT"):
        for ref in _iter_named_inserts_recursive(top, arrow_defs, max_depth=8):
            name = ref.dxf.name or ""
            segs = _get_lines_from_insert(ref)

            if len(segs) != 8:
                summary["shape_fail_not_8_lines_at_runtime"] += 1
                continue

            shaft = _longest_segment(segs)
            if shaft is None:
                summary["shape_fail_no_shaft"] += 1
                continue

            p, q = shaft
            direction_objects.append(
                Direction(
                    startPoint=(round(q[0]), round(q[1])),
                    endPoint=(round(p[0]), round(p[1])),
                )
            )
            per_block_counter[name] += 1
            summary["arrow_ok"] += 1

    return summary, per_block_counter, arrow_defs, defs_debug

def ask_existing_dxf(prompt="DXF 파일 이름: "):
    while True:
        #s = input(prompt).strip().strip('"').strip("'")
        # if not s:
        #     print("⚠️ 비어있습니다. 파일명을 입력하세요.")
        #     continue
        #s = "CadToMap_Input/" + s
        s = "test.dxf"
        p = Path(s)
        if p.suffix.lower() != ".dxf":
            p = p.with_suffix(".dxf")
        if p.exists():
            return p
        else:
            print(f"❌ 파일이 없습니다: {p}")
            try:
                from glob import glob
                print(f"현재 작업 폴더: {Path.cwd()}")
                print("여기 있는 DXF들:", glob("*.dxf"))
            except Exception:
                pass

# ===== 실행 예시 =====
if __name__ == "__main__":

    INPUT_DXF = ask_existing_dxf()
    doc = ezdxf.readfile(INPUT_DXF)

    line_objects, arc_objects = [], []
    eqNode_objects, stbNode_objects = [], []
    direction_objects = []

    # [수정] 레일/아크 수집 로직 (자동 레이어/블록 타겟팅 포함)
    exploded_count, converted_count = collect_rails_into_lists(doc, line_objects, arc_objects)

    # TransferNode 수집
    collect_transferNodes(doc, eqNode_objects, stbNode_objects)

    # 화살표 수집
    dir_summary, per_block_counts, arrow_defs, defs_debug = collect_directions_from_line8_block_inserts(
        doc, direction_objects
    )

    # ===== 결과 출력 =====
    print("\n===== 결과 =====")
    print(f"해체된 INSERT 수(레일 파트): {exploded_count}")
    print(f"해체 결과로 변환된 LINE/ARC 개수: {converted_count}")
    print(f"LINE 객체 수(전체): {len(line_objects)}")
    print(f"ARC  객체 수(전체): {len(arc_objects)}")
    print(f"TransferNode 수(전체): {len(eqNode_objects) + len(stbNode_objects)}")

    # 사용자 요청 출력문 유지
    for arc in arc_objects:
        if arc.center[1] >= 72000 and arc.center[1] <= 75000 and arc.center[0] >= 518000 and arc.center[0] <= 520000:
            print(arc)


🔎 [분석] 도면 내 87개의 레이어가 감지되었습니다. 자동 파악을 진행합니다.

===== 결과 =====
해체된 INSERT 수(레일 파트): 1
해체 결과로 변환된 LINE/ARC 개수: 72
LINE 객체 수(전체): 32
ARC  객체 수(전체): 40
TransferNode 수(전체): 0


In [10]:
# CAD > Map
import math
import copy
import csv

# Line객체를 Node객체, Link객체로 변환
nodes_list = []
links_list = []
horizontalLinks = {}
verticalLinks = {}
ports_list = []
eqNodeIdMatch = {}
stbNodeIdMatch = {}
eqNodes_list = eqNode_objects
stbNodes_list = stbNode_objects
nodeId = 1
linkId = 1

def group_nodes_by_EQ(tmpEqNodes_list, eqNodeIdMatch, ports_list):
    EQ_PORT_NUM = []
    csv_path = Path("CadToMap_Input/EqPort_Num.csv")
    with csv_path.open("r", encoding="utf-8-sig") as f:  # BOM 대비해 utf-8-sig
        for line in f:
            name = line.strip()
            if name:            # 빈 줄은 건너뜀
                EQ_PORT_NUM.append(int(name))
    EQ_PORT_NUM.sort(reverse=True)
    portDiffer = 0
    csv_path = Path("CadToMap_Input/EqPort_Difference.csv")
    with csv_path.open("r", encoding="utf-8-sig") as f:  # BOM 대비해 utf-8-sig
        for line in f:
            name = line.strip()
            if name:            # 빈 줄은 건너뜀
                portDiffer = int(name)
    result_EQs = []
    remaining = list(tmpEqNodes_list)
    alreadyMade = []
    eqCount = 0
    for num in EQ_PORT_NUM:
        for node in remaining:
            if node in alreadyMade:
                continue
            x_matches = [n for n in remaining if n != node and int(n.x) == int(node.x)]
            y_matches = [n for n in remaining if n != node and int(n.y) == int(node.y)]

            x_matches_sorted = sorted(x_matches, key=lambda n: abs(n.y - node.y))
            x_candidates = [node] + x_matches_sorted[:num-1]
            
            ys = sorted(n.y for n in x_candidates)

            if len(ys) == num and int(ys[-1] - ys[0]) <= (num-1)*portDiffer:
                eqCount += 1
                portCount = 1
                
                for n in x_candidates:
                    tmpId = "EQ"+str(eqCount)+"_"+str(portCount)
                    ports_list.append(Port(id=tmpId, type="EQ", carrierType="F400", x=n.x, y=n.y, nodeId=eqNodeIdMatch[(n.x, n.y)], nodeAlignment="U", teachingDone = 0))
                    portCount += 1
                    alreadyMade.append(n)

            
    



def calculatePoint(angle,center,radius):
    revisedAngle = angle
    xMultiplier = 1
    yMultiplier = 1
    newPoint = []
    if angle >= 90 and angle < 180:
        revisedAngle = 180-angle
        xMultiplier = -1
        yMultiplier = 1
    if angle >= 180 and angle < 270:
        revisedAngle = angle-180
        xMultiplier = -1
        yMultiplier = -1
    if angle >= 270 and angle <= 360:
        revisedAngle = 360-angle
        xMultiplier = 1
        yMultiplier = -1
    rad = math.radians(revisedAngle)
    newPoint.append(center[0]+xMultiplier*math.cos(rad)*radius)
    newPoint.append(center[1]+yMultiplier*math.sin(rad)*radius)
    return newPoint


motherNodesList = []
motherLinksList = []
diagonalLinksNodesMatch = {}
diagonalLinksList = []
motherVerticalLinks = {}
motherHorizontalLinks= {}
motherSonLinksMatch = {}
motherStartEndNodesList = []

count = 1 
railGap = 0
csv_path = Path("CadToMap_Input/Rail_Gap.csv")
with csv_path.open("r", encoding="utf-8-sig") as f:  # BOM 대비해 utf-8-sig
    for line in f:
        name = line.strip()
        if name:            # 빈 줄은 건너뜀
            railGap = float(name)
print("직선 Link 변환중...")
for line in line_objects:
    startX = round(line.start[0])
    startY = round(line.start[1])
    endX = round(line.end[0])
    endY = round(line.end[1])
    tmpGNodesList = []
    motherNodesList.append(Node(id=str(nodeId).zfill(6), type="G", reality="R", x=startX, y=startY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None))
    motherNodesList.append(Node(id=str(nodeId).zfill(6), type="G", reality="R", x=endX, y=endY, layer=line.layer,waitNode=None, parentLinkId = None, relativeDistance = None))
    tmpMotherLink = Link(id=str(linkId), type="S", startNode=motherNodesList[len(motherNodesList)-2], endNode=motherNodesList[len(motherNodesList)-1], length = None)
    motherLinksList.append(tmpMotherLink)
    motherSonLinksMatch[motherLinksList[len(motherLinksList)-1]] = []
    if startX == endX:
        if startX in motherVerticalLinks:
            motherVerticalLinks[startX].append(motherLinksList[len(motherLinksList)-1])
        else:
            motherVerticalLinks[startX] = [motherLinksList[len(motherLinksList)-1]]
        m = 1
        if startY > endY:
            m = -1
        currentY = startY
        numNode = 0
        realRailGap = 0
        if max(startY, endY) - min(startY, endY) > 2*railGap:
            numNode = (max(startY,endY)-min(startY,endY))//railGap
            realRailGap = (max(startY, endY) - min(startY, endY))/numNode
        else:
            realRailGap = max(startY, endY) - min(startY, endY) + 100
        while m*currentY < m*endY:
            tmpGNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=startX, y=currentY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None)
            tmpGNodesList.append(tmpGNode)
            nodes_list.append(tmpGNode)
            nodeId += 1
            currentY += m*realRailGap
        tmpGNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=endX, y=endY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None)
        tmpGNodesList.append(tmpGNode)
        nodes_list.append(tmpGNode)
        nodeId += 1
        motherStartEndNodesList.append(tmpGNodesList[0])
        motherStartEndNodesList.append(tmpGNodesList[len(tmpGNodesList)-1])
        for nodeCount in range(len(tmpGNodesList)-1):
            tmpLink = Link(id=str(linkId), type="S", startNode=tmpGNodesList[nodeCount], endNode=tmpGNodesList[nodeCount+1], length = None)
            tmpLink.length = math.hypot(tmpLink.endNode.x-tmpLink.startNode.x, tmpLink.endNode.y-tmpLink.startNode.y)
            links_list.append(tmpLink)
            linkId += 1
            if startX in verticalLinks:
                verticalLinks[startX].append(tmpLink)
            else:
                verticalLinks[startX] = [tmpLink]
            motherSonLinksMatch[motherLinksList[len(motherLinksList)-1]].append(tmpLink)
    elif startY == endY:
        if startY in motherHorizontalLinks:
            motherHorizontalLinks[startY].append(motherLinksList[len(motherLinksList)-1])
        else:
            motherHorizontalLinks[startY] = [motherLinksList[len(motherLinksList)-1]]
        m = 1
        if startX > endX:
            m = -1
        currentX = startX
        numNode = 0
        realRailGap = 0
        if max(startX, endX) - min(startX, endX) > 2*railGap:
            numNode = (max(startX,endX)-min(startX,endX))//railGap
            realRailGap = (max(startX, endX) - min(startX, endX))/numNode
        else:
            realRailGap = max(startX, endX) - min(startX, endX) + 100
        while m*currentX < m*endX:
            tmpGNodesList.append(Node(id=str(nodeId).zfill(6), type="G", reality="R", x=currentX, y=startY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None))
            nodes_list.append(tmpGNodesList[len(tmpGNodesList)-1])
            nodeId += 1
            currentX += m*realRailGap
        tmpGNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=endX, y=endY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None)
        tmpGNodesList.append(tmpGNode)
        nodes_list.append(tmpGNode)
        nodeId += 1
        motherStartEndNodesList.append(tmpGNodesList[0])
        motherStartEndNodesList.append(tmpGNodesList[len(tmpGNodesList)-1])
        for nodeCount in range(len(tmpGNodesList)-1):
            tmpLink = Link(id=str(linkId), type="S", startNode=tmpGNodesList[nodeCount], endNode=tmpGNodesList[nodeCount+1], length = None)
            tmpLink.length = math.hypot(tmpLink.endNode.x-tmpLink.startNode.x, tmpLink.endNode.y-tmpLink.startNode.y)
            links_list.append(tmpLink)
            linkId += 1
            if startY in horizontalLinks:
                horizontalLinks[startY].append(tmpLink)
            else:
                horizontalLinks[startY] = [tmpLink]
            motherSonLinksMatch[motherLinksList[len(motherLinksList)-1]].append(tmpLink)
    else:
        nodes_list.append(Node(id=str(nodeId).zfill(6), type="G", reality="R", x=startX, y=startY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None))
        nodeId += 1
        nodes_list.append(Node(id=str(nodeId).zfill(6), type="G", reality="R", x=endX, y=endY, layer=line.layer, waitNode=None, parentLinkId = None, relativeDistance = None))
        nodeId += 1
        tmpLink = Link(id=str(linkId), type="S", startNode=nodes_list[len(nodes_list)-2], endNode=nodes_list[len(nodes_list)-1], length = None)
        tmpLink.length = math.hypot(tmpLink.endNode.x-tmpLink.startNode.x, tmpLink.endNode.y-tmpLink.startNode.y)
        links_list.append(tmpLink)
        linkId += 1
        diagonalLinksNodesMatch[nodes_list[len(nodes_list)-2]] = links_list[len(links_list)-1]
        diagonalLinksNodesMatch[nodes_list[len(nodes_list)-1]] = links_list[len(links_list)-1]
        diagonalLinksList.append(tmpLink)
        # print(f"link: {links_list[len(links_list)-1].id}, start x: {links_list[len(links_list)-1].startNode.x}, start y: {links_list[len(links_list)-1].startNode.y}, end x: {links_list[len(links_list)-1].endNode.x}, end y: {links_list[len(links_list)-1].endNode.y}")
# print(motherVerticalLinks) 
# print(motherHorizontalLinks)            
# print(verticalLinks)
# print(horizontalLinks)
    #-------------------------------------------------Link 방향성 파악-----------------------------------------------------
searchRange = (0, 1, -1)
for direction in direction_objects:
    if direction.startPoint[0] == direction.endPoint[0]:
        for i in searchRange:
            if direction.startPoint[0]+i in motherVerticalLinks:
                for motherLink in motherVerticalLinks[direction.startPoint[0]+i]:
                    maxY = max(direction.startPoint[1], direction.endPoint[1])
                    minY = min(direction.startPoint[1], direction.endPoint[1])
                    maxLinkY = max(motherLink.startNode.y, motherLink.endNode.y)
                    minLinkY = min(motherLink.startNode.y, motherLink.endNode.y)
                    if minY > minLinkY and minY < maxLinkY:
                        if maxY == direction.startPoint[1] and maxLinkY == motherLink.startNode.y:
                            continue
                        if minY == direction.startPoint[1] and minLinkY == motherLink.startNode.y:
                            continue
                        else:
                            # print("--1--")
                            tmpStartNode = motherLink.startNode
                            motherLink.startNode = motherLink.endNode
                            motherLink.endNode = tmpStartNode
                            for sonLink in motherSonLinksMatch[motherLink]:
                                tmpSonStartNode = sonLink.startNode
                                sonLink.startNode = sonLink.endNode
                                sonLink.endNode = tmpSonStartNode
                    elif maxY > minLinkY and maxY < maxLinkY:
                        if maxY == direction.startPoint[1] and maxLinkY == motherLink.startNode.y:
                            continue
                        if minY == direction.startPoint[1] and minLinkY == motherLink.startNode.y:
                            continue
                        else:
                            # print("--1--")
                            tmpStartNode = motherLink.startNode
                            motherLink.startNode = motherLink.endNode
                            motherLink.endNode = tmpStartNode
                            for sonLink in motherSonLinksMatch[motherLink]:
                                tmpSonStartNode = sonLink.startNode
                                sonLink.startNode = sonLink.endNode
                                sonLink.endNode = tmpSonStartNode

    else:
        for i in range (-1,2):
            if direction.startPoint[1]+i in motherHorizontalLinks:
                for motherLink in motherHorizontalLinks[direction.startPoint[1]+i]:
                    maxX = max(direction.startPoint[0], direction.endPoint[0])
                    minX = min(direction.startPoint[0], direction.endPoint[0])
                    maxLinkX = max(motherLink.startNode.x, motherLink.endNode.x)
                    minLinkX = min(motherLink.startNode.x, motherLink.endNode.x)
                    if minX > minLinkX and minX < maxLinkX:
                        if maxX == direction.startPoint[0] and maxLinkX == motherLink.startNode.x:
                            continue
                        if minX == direction.startPoint[0] and minLinkX == motherLink.startNode.x:
                            continue
                        else:
                            # print("--2--")
                            tmpStartNode = motherLink.startNode
                            motherLink.startNode = motherLink.endNode
                            motherLink.endNode = tmpStartNode
                            for sonLink in motherSonLinksMatch[motherLink]:
                                tmpSonStartNode = sonLink.startNode
                                sonLink.startNode = sonLink.endNode
                                sonLink.endNode = tmpSonStartNode
                    if maxX > minLinkX and maxX < maxLinkX:
                        if maxX == direction.startPoint[0] and maxLinkX == motherLink.startNode.x:
                            continue
                        if minX == direction.startPoint[0] and minLinkX == motherLink.startNode.x:
                            continue
                        else:
                            # print("--2--")
                            tmpStartNode = motherLink.startNode
                            motherLink.startNode = motherLink.endNode
                            motherLink.endNode = tmpStartNode
                            for sonLink in motherSonLinksMatch[motherLink]:
                                tmpSonStartNode = sonLink.startNode
                                sonLink.startNode = sonLink.endNode
                                sonLink.endNode = tmpSonStartNode

    if INPUT_DXF == "test.dxf": 
        for link in horizontalLinks[106736]:
            if link.startNode.x >= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[105736]:
            if link.startNode.x <= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[56686]:
            if link.startNode.x >= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[55686]:
            if link.startNode.x <= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[57586]:
            if link.startNode.x >= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[56586]:
            if link.startNode.x <= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[105836]:
            if link.startNode.x >= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode
        for link in horizontalLinks[104836]:
            if link.startNode.x <= link.endNode.x:
                tmpNode = link.endNode
                link.endNode = link.startNode
                link.startNode = tmpNode




mergeSameCount = 0
searchRange = (0,1,-1)
nCandidates = {}
print("곡선 Link 변환중...")
#Arc객체를 Link객체, Node객체로 변환
for arc in arc_objects:
    arcDirectionFound = False
    arcType = "L"
    startNode = None
    endNode = None
    mergeNode1 = None
    mergeNode2 = None
    startNodeAlreadyMade = False
    endNodeAlreadyMade = False
    arcMirrored = False
    startPoint = calculatePoint(arc.start_angle, arc.center, arc.radius)
    endPoint = calculatePoint(arc.end_angle, arc.center, arc.radius)
    for node in nodes_list:
        for i in searchRange:
            for j in searchRange:
                if node.x == round(startPoint[0],0)+i and node.y == round(startPoint[1],0)+j:
                    startNode = node
                    startNodeAlreadyMade = True
                elif node.x == round(endPoint[0],0)+i and node.y == round(endPoint[1],0)+j:
                    endNode = node
                    endNodeAlreadyMade = True
                if startNodeAlreadyMade == True and endNodeAlreadyMade == True:
                    break
            if startNodeAlreadyMade == True and endNodeAlreadyMade == True:
                    break
        if startNodeAlreadyMade == True and endNodeAlreadyMade == True:
            break
    # print(startNode)
    # print(endNode)
    # print(arc.center)
    # print(arc.start_angle)
    # print(arc.end_angle)
    if arc.end_angle == 0:
        arc.end_angle = 360
    if startNode is None:
        breakLine = False
        if abs(arc.end_angle - arc.start_angle) < 90:
            # print(arc)
            # print("startNodeNone") 
            pass
        startNode = Node(id = str(nodeId).zfill(6), type = "G", reality = "R", x = round(startPoint[0],0), y = round(startPoint[1],0), layer=arc.layer, waitNode=None, parentLinkId = None, relativeDistance = None)
        nodes_list.append(startNode)
        nodeId += 1
        for i in searchRange:
            if startNode.x+i in verticalLinks:
                for link in verticalLinks[startNode.x+i]:
                    maxY = max(link.startNode.y, link.endNode.y)
                    minY = min(link.startNode.y, link.endNode.y)
                    if startNode.y > minY and startNode.y < maxY and link.startNode.layer == startNode.layer:
                        if link.startNode in motherStartEndNodesList and abs(link.startNode.y - startNode.y) < 100:
                            startNode.x = startNode.x+i
                            startNode.id = link.startNode.id
                            nodes_list.remove(link.startNode)
                            link.startNode = startNode
                            breakLine = True
                            break
                        elif link.endNode in motherStartEndNodesList and abs(link.endNode.y - startNode.y) < 100:
                            startNode.x = startNode.x+i
                            startNode.id = link.endNode.id
                            nodes_list.remove(link.endNode)
                            link.endNode = startNode
                            break
                        else:
                            startNode.x = startNode.x+i
                            links_list.append(Link(id=link.id, type="S", startNode=link.startNode, endNode=startNode, length = math.hypot(startNode.x-link.startNode.x, startNode.y-link.startNode.y)))
                            verticalLinks[startNode.x].append(links_list[len(links_list)-1])
                            links_list.append(Link(id=str(linkId), type="S", startNode=startNode, endNode=link.endNode, length = math.hypot(link.endNode.x-startNode.x, link.endNode.y-startNode.y)))
                            linkId += 1
                            verticalLinks[startNode.x].append(links_list[len(links_list)-1])
                            links_list.remove(link)
                            verticalLinks[startNode.x].remove(link)
                            breakLine = True
                            break
            if breakLine == True:
                break
        for i in searchRange:
            if startNode.y+i in horizontalLinks:
                for link in horizontalLinks[startNode.y+i]:
                    maxX = max(link.startNode.x, link.endNode.x)
                    minX = min(link.startNode.x, link.endNode.x)
                    if startNode.x > minX and startNode.x < maxX and link.startNode.layer == startNode.layer:
                        if link.startNode in motherStartEndNodesList and abs(link.startNode.x - startNode.x) < 100:
                            startNode.y = startNode.y+i
                            startNode.id = link.startNode.id
                            nodes_list.remove(link.startNode)
                            link.startNode = startNode
                            breakLine = True
                            break
                        elif link.endNode in motherStartEndNodesList and abs(link.endNode.x - startNode.x) < 100:
                            startNode.y = startNode.y+i
                            startNode.id = link.endNode.id
                            nodes_list.remove(link.endNode)
                            link.endNode = startNode
                            break
                        else:
                            startNode.y = startNode.y+i
                            links_list.append(Link(id=link.id, type="S", startNode=link.startNode, endNode=startNode, length = math.hypot(startNode.x-link.startNode.x, startNode.y-link.startNode.y)))
                            horizontalLinks[startNode.y].append(links_list[len(links_list)-1])
                            links_list.append(Link(id=str(linkId), type="S", startNode=startNode, endNode=link.endNode, length = math.hypot(link.endNode.x-startNode.x, link.endNode.y-startNode.y)))
                            linkId += 1
                            horizontalLinks[startNode.y].append(links_list[len(links_list)-1])
                            links_list.remove(link)
                            horizontalLinks[startNode.y].remove(link)
                            breakLine = True
                            break
            if breakLine == True:
                    break
    if endNode is None:
        if abs(arc.end_angle - arc.start_angle) < 90:
            # print(arc)
            # print("endNodeNone") 
            pass

        endNode = Node(id = str(nodeId).zfill(6), type = "G", reality = "R", x = round(endPoint[0],0), y = round(endPoint[1],0), layer = arc.layer, waitNode=None, parentLinkId = None, relativeDistance = None)
        nodes_list.append(endNode)
        nodeId += 1
        breakLine = False
        for i in searchRange:
            if endNode.x+i in verticalLinks:
                for link in verticalLinks[endNode.x+i]:
                    maxY = max(link.startNode.y, link.endNode.y)
                    minY = min(link.startNode.y, link.endNode.y)
                    if endNode.y > minY and endNode.y < maxY and link.startNode.layer == endNode.layer:
                        if link.startNode in motherStartEndNodesList and abs(link.startNode.y - endNode.y) < 100:
                            endNode.x = endNode.x+i
                            endNode.id = link.startNode.id
                            nodes_list.remove(link.startNode)
                            link.startNode = endNode
                            breakLine = True
                            break
                        elif link.endNode in motherStartEndNodesList and abs(link.endNode.y - endNode.y) < 100:
                            endNode.x = endNode.x+i
                            endNode.id = link.endNode.id
                            nodes_list.remove(link.endNode)
                            link.endNode = endNode
                            break
                        else:
                            endNode.x = endNode.x+i
                            links_list.append(Link(id=link.id, type="S", startNode=link.startNode, endNode=endNode, length = math.hypot(endNode.x-link.startNode.x, endNode.y-link.startNode.y)))
                            verticalLinks[endNode.x].append(links_list[len(links_list)-1])
                            links_list.append(Link(id=str(linkId), type="S", startNode=endNode, endNode=link.endNode, length = math.hypot(link.endNode.x-endNode.x, link.endNode.y-endNode.y)))
                            linkId += 1
                            verticalLinks[endNode.x].append(links_list[len(links_list)-1])
                            links_list.remove(link)
                            verticalLinks[endNode.x].remove(link)
                            breakLine = True
                            break
            if breakLine == True:
                break
        for i in searchRange:
            if endNode.y+i in horizontalLinks:
                for link in horizontalLinks[endNode.y+i]:
                    maxX = max(link.startNode.x, link.endNode.x)
                    minX = min(link.startNode.x, link.endNode.x)
                    if endNode.x > minX and endNode.x < maxX and link.startNode.layer == endNode.layer:
                        if link.startNode in motherStartEndNodesList and abs(link.startNode.x - endNode.x) < 100:
                            endNode.y = endNode.y+i
                            endNode.id = link.startNode.id
                            nodes_list.remove(link.startNode)
                            link.startNode = endNode
                            breakLine = True
                            break
                        elif link.endNode in motherStartEndNodesList and abs(link.endNode.x - endNode.x) < 100:
                            endNode.y = endNode.y+i
                            endNode.id = link.endNode.id
                            nodes_list.remove(link.endNode)
                            link.endNode = endNode
                            break
                        else:
                            endNode.y = endNode.y+i
                            links_list.append(Link(id=link.id, type="S", startNode=link.startNode, endNode=endNode, length = math.hypot(endNode.x-link.startNode.x, endNode.y-link.startNode.y)))
                            horizontalLinks[endNode.y].append(links_list[len(links_list)-1])
                            links_list.append(Link(id=str(linkId), type="S", startNode=endNode, endNode=link.endNode, length = math.hypot(link.endNode.x-endNode.x, link.endNode.y-endNode.y)))
                            linkId += 1
                            horizontalLinks[endNode.y].append(links_list[len(links_list)-1])
                            links_list.remove(link)
                            horizontalLinks[endNode.y].remove(link)
                            breakLine=True
                            break
            if breakLine == True:
                break
    if startNodeAlreadyMade==True and endNodeAlreadyMade==True and arc.end_angle-arc.start_angle==90:
        if startNode.x in verticalLinks:
            for link in verticalLinks[startNode.x]:
                if link.startNode == startNode:
                    tmpNode = endNode
                    endNode = startNode
                    startNode = tmpNode
                    arcType = "R"
        if startNode.y in horizontalLinks:
            for link in horizontalLinks[startNode.y]:
                if link.startNode == startNode:
                    tmpNode = endNode
                    endNode = startNode
                    startNode = tmpNode
                    arcType = "R"
        arcDirectionFound = True
    if abs(arc.end_angle - arc.start_angle) == 180:
        if startNode.x in motherVerticalLinks:
            startNodeMotherLinkDirection = 0 
            arcDirection = 0
            for link in motherVerticalLinks[startNode.x]:
                maxLinkY = max(link.startNode.y, link.endNode.y)
                minLinkY = min(link.startNode.y, link.endNode.y)
                if startNode.y <= maxLinkY and startNode.y >= minLinkY and link.startNode.layer == startNode.layer:
                    if maxLinkY == link.startNode.y:
                        startNodeMotherLinkDirection = -1
                        break
                    else:
                        startNodeMotherLinkDirection = 1
                        break
            if arc.start_angle == 0:
                arcDirection = 1
            elif arc.start_angle == 180:
                arcDirection = -1
            if startNodeMotherLinkDirection != arcDirection:
                tmpNode = endNode
                endNode = startNode
                startNode = tmpNode
                arcType ="R"
            if startNodeMotherLinkDirection == 0 or arcDirection == 0:
                # print("ERROR 1 !!!")
                pass
        elif startNode.y in motherHorizontalLinks:
            startNodeMotherLinkDirection = 0 
            arcDirection = 0
            for link in motherHorizontalLinks[startNode.y]:
                maxLinkX = max(link.startNode.x, link.endNode.x)
                minLinkX = min(link.startNode.x, link.endNode.x)
                if startNode.x <= maxLinkX and startNode.x >= minLinkX:
                    if maxLinkX == link.startNode.x:
                        startNodeMotherLinkDirection = -1
                        break
                    else:
                        startNodeMotherLinkDirection = 1
                        break
            if arc.start_angle == 270:
                arcDirection = 1
            elif arc.start_angle == 90:
                arcDirection = -1
            if startNodeMotherLinkDirection != arcDirection:
                tmpNode = endNode
                endNode = startNode
                startNode = tmpNode
                arcType = "R"
            if startNodeMotherLinkDirection == 0 or arcDirection == 0:
                # print("ERROR 2 !!!")
                pass
        arcDirectionFound = True

    if arcDirectionFound == False and abs(arc.end_angle-arc.start_angle) < 90:
        if startNode.x in motherVerticalLinks:
            motherLinkDirection = 0
            arcDirection = 0 
            realyInLink = False
            for link in motherVerticalLinks[startNode.x]:
                maxLinkY = max(link.startNode.y, link.endNode.y)
                minLinkY = min(link.startNode.y, link.endNode.y)
                if startNode.y <= maxLinkY and startNode.y >= minLinkY:
                    if maxLinkY == link.startNode.y:
                        motherLinkDirection = -1
                    else:
                        motherLinkDirection = 1
                    realyInLink = True
                    break
            if realyInLink == True:
                if startNode.y > endNode.y:
                    arcDirection = -1
                else:
                    arcDirection = 1
                if arcDirection != motherLinkDirection:
                    tmpNode = endNode
                    endNode = startNode                                                                                                             
                    startNode = tmpNode
                    arcType = "R"
                    if diagonalLinksNodesMatch[startNode].endNode != startNode:
                        tmpNode = diagonalLinksNodesMatch[startNode].startNode
                        diagonalLinksNodesMatch[startNode].startNode = diagonalLinksNodesMatch[startNode].endNode
                        diagonalLinksNodesMatch[startNode].endNode = tmpNode
        if endNode.x in motherVerticalLinks:
            motherLinkDirection = 0
            arcDirection = 0 
            realyInLink = False
            for link in motherVerticalLinks[endNode.x]:
                maxLinkY = max(link.startNode.y, link.endNode.y)
                minLinkY = min(link.startNode.y, link.endNode.y)
                if endNode.y <= maxLinkY and endNode.y >= minLinkY:
                    if maxLinkY == link.startNode.y:
                        motherLinkDirection = -1
                    else:
                        motherLinkDirection = 1
                    realyInLink = True
                    break
            if realyInLink == True:
                if startNode.y > endNode.y:
                    arcDirection = -1
                else:
                    arcDirection = 1
                if arcDirection != motherLinkDirection:
                    tmpNode = endNode
                    endNode = startNode
                    startNode = tmpNode
                    arcType = "R"
                    if diagonalLinksNodesMatch[endNode].startNode != endNode:
                        tmpNode = diagonalLinksNodesMatch[endNode].endNode
                        diagonalLinksNodesMatch[endNode].endNode = diagonalLinksNodesMatch[endNode].startNode
                        diagonalLinksNodesMatch[endNode].startNode = tmpNode
        if startNode.y in motherHorizontalLinks:
            motherLinkDirection = 0 
            arcDirection = 0
            realyInLink = False
            for link in motherHorizontalLinks[startNode.y]:
                maxLinkX = max(link.startNode.x, link.endNode.x)
                minLinkX = min(link.startNode.x, link.endNode.x)
                if startNode.x <= maxLinkX and startNode.x >= minLinkX:
                    if maxLinkX == link.startNode.x:
                        motherLinkDirection = -1
                    else:
                        motherLinkDirection = 1
                    realyInLink = True
                    break
            if realyInLink == True:
                if startNode.x > endNode.x:
                    arcDirection = -1
                else:
                    arcDirection = 1
                if arcDirection != motherLinkDirection:
                    tmpNode = endNode
                    endNode = startNode
                    startNode = tmpNode
                    arcType = "R"
                    if diagonalLinksNodesMatch[startNode].endNode != startNode:
                        tmpNode = diagonalLinksNodesMatch[startNode].startNode
                        diagonalLinksNodesMatch[startNode].startNode = diagonalLinksNodesMatch[startNode].endNode
                        diagonalLinksNodesMatch[startNode].endNode = tmpNode
        if endNode.y in motherHorizontalLinks:
            motherLinkDirection = 0 
            arcDirection = 0
            realyInLink = False
            for link in motherHorizontalLinks[endNode.y]:
                maxLinkX = max(link.startNode.x, link.endNode.x)
                minLinkX = min(link.startNode.x, link.endNode.x)
                if endNode.x <= maxLinkX and endNode.x >= minLinkX:
                    if maxLinkX == link.startNode.x:
                        motherLinkDirection = -1
                    else:
                        motherLinkDirection = 1
                    realyInLink = True
                    break
            if realyInLink == True:
                if startNode.x > endNode.x:
                    arcDirection = -1
                else:
                    arcDirection = 1
                if arcDirection != motherLinkDirection:
                    tmpNode = endNode
                    endNode = startNode
                    startNode = tmpNode
                    arcType = "R"
                    if diagonalLinksNodesMatch[endNode].startNode != endNode:
                        tmpNode = diagonalLinksNodesMatch[endNode].endNode
                        diagonalLinksNodesMatch[endNode].endNode = diagonalLinksNodesMatch[endNode].startNode
                        diagonalLinksNodesMatch[endNode].startNode = tmpNode
    arc90mirrored = False
    if arcDirectionFound == False and abs(arc.end_angle-arc.start_angle) == 90:
        if startNode.x in verticalLinks:
            for link in verticalLinks[startNode.x]:
                maxLinkY = max(link.startNode.y, link.endNode.y)
                minLinkY = min(link.startNode.y, link.endNode.y)
                if maxLinkY >= startNode.y and minLinkY <= startNode.y:
                    if link.startNode.y == maxLinkY and startNode.y < endNode.y:
                        tmpNode = endNode
                        endNode = startNode
                        startNode = tmpNode
                        arcType = "R"
                        arc90mirrored = True
                        
                    elif link.startNode.y == minLinkY and startNode.y > endNode.y:
                        tmpNode = endNode
                        endNode = startNode
                        startNode = tmpNode
                        arcType = "R"
                        arc90mirrored = True
                        
        elif startNode.y in horizontalLinks:
            for link in horizontalLinks[startNode.y]:
                maxLinkX = max(link.startNode.x, link.endNode.x)
                minLinkX = min(link.startNode.x, link.endNode.x)
                if maxLinkX >= startNode.x and minLinkX <= startNode.x:
                    if link.startNode.x == maxLinkX and startNode.x < endNode.x:
                        tmpNode = endNode
                        endNode = startNode
                        startNode = tmpNode
                        arcType = "R"
                        arc90mirrored = True
                    elif link.startNode.x == minLinkX and startNode.x > endNode.x:
                        tmpNode = endNode
                        endNode = startNode
                        startNode = tmpNode
                        arcType = "R"
                        arc90mirrored = True

    
                
    # print("Finished")
    if abs(arc.start_angle-arc.end_angle) == 180:
        arcType = "U"
    delta = (arc.end_angle - arc.start_angle) % 360.0
    length = arc.radius*math.radians(delta)
    if arcType == "R" and startNode.x > endNode.x: # (예시 조건: 실제 도면 상황에 맞춰 방향 조건 수정 필요)
        # R을 L로 바꾸고 노드 순서를 뒤집음
        arcType = "L"
        startNode, endNode = endNode, startNode
    elif arcType == "L" and startNode.x > endNode.x:
        arcType = "R"
        startNode, endNode = endNode, startNode
    tmpLink = Link(id=str(linkId), type = arcType, startNode = startNode, endNode = endNode, length = length)
    links_list.append(tmpLink)
    if startNode in diagonalLinksNodesMatch:
        if diagonalLinksNodesMatch[startNode].id in nCandidates:
            nCandidates[diagonalLinksNodesMatch[startNode].id].append(tmpLink)
        else:
            nCandidates[diagonalLinksNodesMatch[startNode].id] = [tmpLink, diagonalLinksNodesMatch[startNode]]
    elif endNode in diagonalLinksNodesMatch:
        if diagonalLinksNodesMatch[endNode].id in nCandidates:
            nCandidates[diagonalLinksNodesMatch[endNode].id].append(tmpLink)
        else:
            nCandidates[diagonalLinksNodesMatch[endNode].id] = [tmpLink, diagonalLinksNodesMatch[endNode]]
    linkId += 1


    breakLine = False
    for link in motherVerticalLinks.get(startNode.x,[]):
        maxLinkY = max(link.startNode.y, link.endNode.y)
        minLinkY = min(link.startNode.y, link.endNode.y)
        if maxLinkY > startNode.y and minLinkY < startNode.y and link.startNode.layer == startNode.layer:
            m = 0
            if maxLinkY == link.startNode.y:
                m = 1
            else:
                m = -1
            for sonLink in verticalLinks.get(startNode.x,[]):
                maxSonLinkY = max(sonLink.startNode.y, sonLink.endNode.y)
                minSonLinkY = min(sonLink.startNode.y, sonLink.endNode.y)
                if sonLink.startNode.y == startNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
                elif sonLink.endNode.y == startNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.endNode.setWaitNode("S")
                    breakLine = True
                    break
                elif maxSonLinkY > startNode.y + m*500 and minSonLinkY < startNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    tmpWaitNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=startNode.x, y=startNode.y+m*500, layer=arc.layer, waitNode="S", parentLinkId = None, relativeDistance = None)
                    nodeId += 1
                    nodes_list.append(tmpWaitNode)
                    links_list.append(Link(id=sonLink.id, type="S", startNode=sonLink.startNode, endNode=tmpWaitNode, length = math.hypot(tmpWaitNode.x-sonLink.startNode.x, tmpWaitNode.y-sonLink.startNode.y)))
                    links_list.append(Link(id=str(linkId), type="S", startNode=tmpWaitNode, endNode=sonLink.endNode, length = math.hypot(sonLink.endNode.x-tmpWaitNode.x, sonLink.endNode.y-tmpWaitNode.y)))
                    linkId += 1
                    verticalLinks[startNode.x].append(links_list[len(links_list)-2])
                    verticalLinks[startNode.x].append(links_list[len(links_list)-1])
                    links_list.remove(sonLink)
                    verticalLinks[startNode.x].remove(sonLink)
                    breakLine = True
                    break
                if m == 1 and maxLinkY == maxSonLinkY and maxLinkY < startNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
                elif m == -1 and minLinkY == minSonLinkY and minLinkY > startNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
        if breakLine == True:
                break
    for link in motherVerticalLinks.get(endNode.x,[]):
        maxLinkY = max(link.startNode.y, link.endNode.y)
        minLinkY = min(link.startNode.y, link.endNode.y)
        if maxLinkY > endNode.y and minLinkY < endNode.y and link.startNode.layer == endNode.layer:
            m = 0
            if maxLinkY == link.startNode.y:
                m = 1
            else:
                m = -1
            for sonLink in verticalLinks.get(endNode.x,[]):
                maxSonLinkY = max(sonLink.startNode.y, sonLink.endNode.y)
                minSonLinkY = min(sonLink.startNode.y, sonLink.endNode.y)
                if sonLink.startNode.y == endNode.y + m*500 and sonLink.startNode.layer == endNode.layer:
                    sonLink.startNode.setWaitNode("M")
                    breakLine = True
                    break
                elif sonLink.endNode.y == endNode.y + m*500 and sonLink.startNode.layer == endNode.layer:
                    sonLink.endNode.setWaitNode("M")
                    breakLine = True
                    break
                elif maxSonLinkY > endNode.y + m*500 and minSonLinkY < endNode.y + m*500 and sonLink.startNode.layer == endNode.layer:
                    tmpWaitNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=endNode.x, y=endNode.y+m*500, layer=arc.layer, waitNode="M", parentLinkId = None, relativeDistance = None)
                    nodeId += 1
                    nodes_list.append(tmpWaitNode)
                    links_list.append(Link(id=sonLink.id, type="S", startNode=sonLink.startNode, endNode=tmpWaitNode, length = math.hypot(tmpWaitNode.x-sonLink.startNode.x, tmpWaitNode.y-sonLink.startNode.y)))
                    links_list.append(Link(id=str(linkId), type="S", startNode=tmpWaitNode, endNode=sonLink.endNode, length = math.hypot(sonLink.endNode.x-tmpWaitNode.x, sonLink.endNode.y-tmpWaitNode.y)))
                    linkId += 1
                    verticalLinks[endNode.x].append(links_list[len(links_list)-2])
                    verticalLinks[endNode.x].append(links_list[len(links_list)-1])
                    links_list.remove(sonLink)
                    verticalLinks[endNode.x].remove(sonLink)
                    breakLine = True
                    break
                if m == 1 and maxLinkY == maxSonLinkY and maxLinkY < endNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("M")
                    breakLine = True
                    break
                elif m == -1 and minLinkY == minSonLinkY and minLinkY > endNode.y + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("M")
                    breakLine = True
                    break
        if breakLine == True:
                break
    for link in motherHorizontalLinks.get(startNode.y,[]):
        maxLinkX = max(link.startNode.x, link.startNode.y)
        minLinkX = min(link.startNode.x, link.startNode.y)
        if maxLinkX > startNode.x and minLinkX < startNode.x and link.startNode.layer == startNode.layer:
            m = 0
            if maxLinkX == link.startNode.x: 
                m = 1
            else:
                m = -1
            for sonLink in horizontalLinks.get(startNode.y,[]):
                maxSonLinkX = max(sonLink.startNode.x, sonLink.endNode.x)
                minSonLinkX = min(sonLink.startNode.x, sonLink.endNode.x)
                if sonLink.startNode.x == startNode.x + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
                elif sonLink.endNode.y == startNode.x + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.endNode.setWaitNode("S")
                    breakLine = True
                    break
                elif maxSonLinkX > startNode.x + m*500 and minSonLinkX < startNode.x + m*500 and sonLink.startNode.layer == startNode.layer:
                    tmpWaitNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=startNode.x + m*500, y=startNode.y, layer=arc.layer, waitNode="S", parentLinkId = None, relativeDistance = None)
                    nodeId += 1
                    nodes_list.append(tmpWaitNode)
                    links_list.append(Link(id=sonLink.id, type="S", startNode=sonLink.startNode, endNode=tmpWaitNode, length = math.hypot(tmpWaitNode.x-sonLink.startNode.x, tmpWaitNode.y-sonLink.startNode.y)))
                    links_list.append(Link(id=str(linkId), type="S", startNode=tmpWaitNode, endNode=sonLink.endNode, length = math.hypot(sonLink.endNode.x-tmpWaitNode.x, sonLink.endNode.y-tmpWaitNode.y)))
                    linkId += 1
                    horizontalLinks[startNode.y].append(links_list[len(links_list)-2])
                    horizontalLinks[startNode.y].append(links_list[len(links_list)-1])
                    links_list.remove(sonLink)
                    horizontalLinks[startNode.y].remove(sonLink)
                    breakLine = True
                    break
                if m == 1 and maxLinkX == maxSonLinkX and maxLinkX < startNode.x + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
                elif m == -1 and minLinkX == minSonLinkX and minLinkX > startNode.x +m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
                
        if breakLine == True:
                break
    for link in motherHorizontalLinks.get(endNode.y,[]):
        maxLinkX = max(link.startNode.x, link.startNode.y)
        minLinkX = min(link.startNode.x, link.startNode.y)
        if maxLinkX > endNode.x and minLinkX < endNode.x and link.startNode.layer == endNode.layer:
            m = 0
            if maxLinkX == link.startNode.x: 
                m = 1
            else:
                m = -1
            for sonLink in horizontalLinks.get(endNode.y,[]):
                maxSonLinkX = max(sonLink.startNode.x, sonLink.endNode.x)
                minSonLinkX = min(sonLink.startNode.x, sonLink.endNode.x)
                if sonLink.startNode.x == endNode.x + m*500 and sonLink.endNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("S")
                    breakLine = True
                    break
                elif sonLink.endNode.y == endNode.x + m*500 and sonLink.startNode.layer == endNode.layer:
                    sonLink.endNode.setWaitNode("S")
                    breakLine = True
                    break
                elif maxSonLinkX > endNode.x + m*500 and minSonLinkX < endNode.x + m*500 and sonLink.startNode.layer == endNode.layer:
                    tmpWaitNode = Node(id=str(nodeId).zfill(6), type="G", reality="R", x=endNode.x + m*500, y=endNode.y, layer=arc.layer, waitNode="S", parentLinkId = None, relativeDistance = None)
                    nodeId += 1
                    nodes_list.append(tmpWaitNode)
                    links_list.append(Link(id=sonLink.id, type="S", startNode=sonLink.startNode, endNode=tmpWaitNode, length = math.hypot(tmpWaitNode.x-sonLink.startNode.x, tmpWaitNode.y-sonLink.startNode.y)))
                    links_list.append(Link(id=str(linkId), type="S", startNode=tmpWaitNode, endNode=sonLink.endNode, length = math.hypot(sonLink.endNode.x-tmpWaitNode.x, sonLink.endNode.y-tmpWaitNode.y)))
                    linkId += 1
                    horizontalLinks[endNode.y].append(links_list[len(links_list)-2])
                    horizontalLinks[endNode.y].append(links_list[len(links_list)-1])
                    links_list.remove(sonLink)
                    horizontalLinks[endNode.y].remove(sonLink)
                    breakLine = True
                    break
                if m == 1 and maxLinkX == maxSonLinkX and maxLinkX < startNode.x + m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("M")
                    breakLine = True
                    break
                elif m == -1 and minLinkX == minSonLinkX and minLinkX > startNode.x +m*500 and sonLink.startNode.layer == startNode.layer:
                    sonLink.startNode.setWaitNode("M")
                    breakLine = True
                    break
        if breakLine == True:
                break
        




for dl in diagonalLinksList:
    if dl.id in nCandidates:
        tmpList = nCandidates[dl.id]
        arc1 = tmpList[0]
        link = tmpList[1]
        arc2 = tmpList[2]
        nStartNode = None
        nEndNode = None
        if arc1.startNode == link.startNode:
            nStartNode = arc1.endNode
            nodes_list.remove(arc1.startNode)
        elif arc1.startNode == link.endNode:
            nEndNode = arc1.endNode
            nodes_list.remove(arc1.startNode)
        elif arc1.endNode == link.startNode:
            nStartNode = arc1.startNode
            nodes_list.remove(arc1.endNode)
        elif arc1.endNode == link.endNode:
            nEndNode == arc1.startNode
            nodes_list.remove(arc1.endNode)
        
        if arc2.startNode == link.startNode:
            nStartNode = arc2.endNode
            nodes_list.remove(arc2.startNode)
        elif arc2.startNode == link.endNode:
            nEndNode = arc2.endNode
            nodes_list.remove(arc2.startNode)
        elif arc2.endNode == link.startNode:
            nStartNode = arc2.startNode
            nodes_list.remove(arc2.endNode)
        elif arc2.endNode == link.endNode:
            nEndNode == arc2.startNode
            nodes_list.remove(arc2.endNode)

        nId = arc1.id
        tmpLink = Link(id=nId, type="N", startNode=nStartNode, endNode=nEndNode, length = arc1.length+link.length+arc2.length)
        links_list.append(tmpLink)
        for link in tmpList:
            links_list.remove(link)

print("EQ Port 변환중...")
for eqNode in eqNodes_list:
    eqNodeFound = False
    for gap in range(0,6):
        for k in range(0,1):
            if k == 0:
                gap = -gap
            if round(eqNode.x,0)+gap in verticalLinks:
                for link in verticalLinks[round(eqNode.x,0)+gap]:
                    maxY = max(link.startNode.y, link.endNode.y)
                    minY = min(link.startNode.y, link.endNode.y)
                    if round(eqNode.y,0) > minY and round(eqNode.y,0) < maxY and link.startNode.layer == 0:
                        eqNode.x = eqNode.x + gap
                        tmpStart = Node(id = str(nodeId).zfill(6), type = "T", reality = "R", x = round(eqNode.x,0), y = round(eqNode.y,0), layer=0, waitNode = None, parentLinkId = None, relativeDistance = None)
                        nodes_list.append(tmpStart)
                        

                        eqNodeIdMatch[(tmpStart.x, tmpStart.y)] = tmpStart.id
                        nodeId += 1
                        tmpStart.parentLinkId = link.id
                        tmpStart.relativeDistance = tmpStart.x-link.startNode.x + tmpStart.y-link.startNode.y
                        eqNodeFound = True
                        break
                    elif round(eqNode.y, 0) == link.startNode.y and link.layer == 0:
                        eqNodeIdMatch[(round(eqNode.x,0), round(eqNode.y,0))] = link.startNode.id
                        eqNodeFound = True
                        break
                    elif round(eqNode.y, 0) == link.endNode.y and link.layer == 0:
                        eqNodeIdMatch[(round(eqNode.x,0), round(eqNode.y,0))] = link.endNode.id
                        eqNodeFound = True
                        break
            if eqNodeFound == True:
                break
            if round(eqNode.y,0)+gap in horizontalLinks:
                for link in horizontalLinks[round(eqNode.y,0)+gap]:
                    maxX = max(link.startNode.x, link.endNode.x)
                    minX = min(link.startNode.x, link.endNode.x)
                    if round(eqNode.x,0) > minX and round(eqNode.x,0) < maxX and link.startNode.layer == 0:
                        eqNode.y = eqNode.y + gap
                        tmpStart = Node(id = str(nodeId).zfill(6), type = "T", reality = "R", x = round(eqNode.x,0), y = round(eqNode.y,0), layer=0, waitNode = None, parentLinkId = None, relativeDistance = None)
                        
                        nodes_list.append(tmpStart)
                        eqNodeIdMatch[(tmpStart.x, tmpStart.y)] = tmpStart.id
                        nodeId += 1
                        tmpStart.parentLinkId = link.id
                        tmpStart.relativeDistance = tmpStart.x-link.startNode.x + tmpStart.y-link.startNode.y
                        eqNodeFound = True
                        break
                    elif round(eqNode.x, 0) == link.startNode.x and link.layer == 0:
                        eqNodeIdMatch[(round(eqNode.x,0), round(eqNode.y,0))] = link.startNode.id
                        eqNodeFound = True
                        break
                    elif round(eqNode.x, 0) == link.endNode.x and link.layer == 0:
                        eqNodeIdMatch[(round(eqNode.x,0), round(eqNode.y,0))] = link.endNode.id
                        eqNodeFound = True
                        break
            if eqNodeFound == True:
                break
        if eqNodeFound == True:
                break
tmpEqNodes_list = list(eqNodes_list)
tmpEqNodes_list.sort(key=lambda node: (node.x, node.y))
group_nodes_by_EQ(tmpEqNodes_list, eqNodeIdMatch, ports_list)


print("STB Port 변환중...")    
stbCount = 1
stbNodeIdMatch = {}
csv_path = Path("CadToMap_Input/STB_Rail_Gap.csv")
stbGap = 0
with csv_path.open("r", encoding="utf-8-sig") as f:  # BOM 대비해 utf-8-sig
    for line in f:
        name = line.strip()
        if name:            # 빈 줄은 건너뜀
            stbGap = int(name)
            csv_path = Path("CadToMap_Input/STB_Rail_Gap.csv")

csv_path = Path("CadToMap_Input/STB_Search_Range.csv")
searchRangeMin = 0
searchRangeMax = 0 
with csv_path.open("r", encoding="utf-8-sig") as f:  # BOM 대비해 utf-8-sig
    for line in f:
        name = line.strip()
        if name:            # 빈 줄은 건너뜀
            if searchRangeMin == 0:
                searchRangeMin = int(name)
            else:
                searchRangeMax = int(name)
stbSearchRange = [stbGap] + [i for i in range(searchRangeMin, searchRangeMax + 1) if i != stbGap]


for stbNode in stbNodes_list:
    # print(f"stbNode: {stbNode.x}, {stbNode.y}")
    portId = "STB" + str(stbCount).zfill(5)
    roundedX = round(stbNode.x, 0)
    roundedY = round(stbNode.y, 0)
    portMade = False
    for diffr in stbSearchRange:
        # print(diffr)
        roundedXPlus = roundedX + diffr
        roundedXMinus = roundedX - diffr
        roundedYPlus = roundedY + diffr
        roundedYMinus = roundedY - diffr
        if roundedXPlus in motherVerticalLinks:
            for en in eqNodes_list:
                if round(en.x,0) == roundedXPlus and round(en.y,0) == roundedY:
                    # print("---1/1---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=eqNodeIdMatch[(en.x, en.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            if portMade == True:
                break
            for sn in stbNodes_list:
                if round(sn.x,0) == roundedXPlus and round(sn.y,0) == roundedY:
                    # print("---1/2---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=stbNodeIdMatch[(sn.x, sn.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            if portMade == True:
                break
            for rail in motherVerticalLinks[roundedXPlus]:
                yMax = max(rail.startNode.y, rail.endNode.y)
                yMin = min(rail.startNode.y, rail.endNode.y)
                nodeAlignmentSide = "F"
                if yMax == rail.startNode.y:
                    nodeAlignmentSide = "R"
                else:
                    nodeAlignmentSide = "L"
                if roundedY >= yMin and roundedY <= yMax:
                    # print("---1---")
                    for link in verticalLinks[roundedXPlus]:
                        maxY = max(link.startNode.y, link.endNode.y)
                        minY = min(link.startNode.y, link.endNode.y)
                        if roundedY > minY and roundedY < maxY and link.startNode.layer == 0:
                            # print("---1/3---")
                            tmpNode = Node(id = str(nodeId).zfill(6), type = "T", reality = "R", x = roundedXPlus, y = roundedY, layer = 0, waitNode = None, parentLinkId = link.id, relativeDistance = roundedXPlus-link.startNode.x + roundedY-link.startNode.y)
                            nodes_list.append(tmpNode)
                            stbNodeIdMatch[(stbNode.x, stbNode.y)] = tmpNode.id
                            nodeId += 1
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=tmpNode.id, nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                            stbCount += 1
                            portMade = True
                        
                            break
                        elif roundedY == link.startNode.y and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.startNode.id, nodeAlignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                        elif roundedY == link.endNode.y and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.endNode.id, nodeAlignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                    break
        if roundedXMinus in motherVerticalLinks:
            for en in eqNodes_list:
                if round(en.x,0) == roundedXMinus and round(en.y,0) == roundedY:
                    # print("---2/1---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=eqNodeIdMatch[(en.x, en.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            if portMade == True:
                break
            for sn in stbNodes_list:
                if round(sn.x,0) == roundedXMinus and round(sn.y,0) == roundedY:
                    # print("---2/2---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=stbNodeIdMatch[(sn.x, sn.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            for rail in motherVerticalLinks[roundedXMinus]:
                yMax = max(rail.startNode.y, rail.endNode.y)
                yMin = min(rail.startNode.y, rail.endNode.y)
                nodeAlignmentSide = "F"
                if yMax == rail.startNode.y:
                    nodeAlignmentSide = "L"
                else:
                    nodeAlignmentSide = "R"
                if roundedY >= yMin and roundedY <= yMax:
                    # print("---2---")
                    if portMade == True:
                        break
                    for link in verticalLinks[roundedXMinus]:
                        maxY = max(link.startNode.y, link.endNode.y)
                        minY = min(link.startNode.y, link.endNode.y)
                        if roundedY >= minY and roundedY <= maxY and link.startNode.layer == 0:
                            # print("---2/3---")
                            tmpNode = Node(id = str(nodeId).zfill(6), type = "T", reality = "R", x = roundedXMinus, y = roundedY, layer = 0, waitNode = None, parentLinkId = link.id, relativeDistance = roundedXMinus-link.startNode.x + roundedY-link.startNode.y)
                            nodes_list.append(tmpNode)
                            nodeId += 1
                            stbNodeIdMatch[(stbNode.x, stbNode.y)] = tmpNode.id
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=tmpNode.id, nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                            stbCount += 1
                            portMade = True
                            
                            break
                        elif roundedY == link.startNode.y and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.startNode.id, nodeAlignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                        elif roundedY == link.endNode.y and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.endNode.id, nodeAlignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                    break
        if roundedYPlus in motherHorizontalLinks:
            for en in eqNodes_list:
                if round(en.x,0) == roundedX and round(en.y,0) == roundedYPlus:
                    # print("---3/1---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=eqNodeIdMatch[(en.x, en.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            if portMade == True:
                break
            for sn in stbNodes_list:
                if round(sn.x,0) == roundedX and round(sn.y,0) == roundedYPlus:
                    # print("---3/2---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=stbNodeIdMatch[(sn.x, sn.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            for rail in motherHorizontalLinks[roundedYPlus]:
                xMax = max(rail.startNode.x, rail.endNode.x)
                xMin = min(rail.startNode.x, rail.endNode.x)
                nodeAlignmentSide = "F"
                if xMax == rail.startNode.y:
                    nodeAlignmentSide = "L"
                else:
                    nodeAlignmentSide = "R"
                if roundedX >= xMin and roundedX <= xMax:
                    # print("---3---")
                    if portMade == True:
                        break
                    for link in horizontalLinks[roundedYPlus]:
                        maxX = max(link.startNode.x, link.endNode.x)
                        minX = min(link.startNode.x, link.endNode.x)
                        if roundedX > minX and roundedX < maxX and link.startNode.layer == 0:
                            # print("---3/3---")
                            tmpNode = Node(id = str(nodeId).zfill(6), type = "T", reality = "R", x = roundedX, y = roundedYPlus, layer = 0, waitNode = None, parentLinkId = link.id, relativeDistance = roundedX-link.startNode.x + roundedYPlus-link.startNode.y)
                            nodeId += 1
                            nodes_list.append(tmpNode)
                            stbNodeIdMatch[(stbNode.x, stbNode.y)] = tmpNode.id
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=tmpNode.id, nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                            stbCount += 1
                            portMade = True
                            
                            break
                        elif roundedX == link.startNode.x and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.startNode.id, nodeAlignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                        elif roundedX == link.endNode.x and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.endNode.id, nodeAlignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                    break
        if roundedYMinus in motherHorizontalLinks:
            for en in eqNodes_list:
                if round(en.x,0) == roundedX and round(en.y,0) == roundedYMinus:
                    # print("---4/1---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=eqNodeIdMatch[(en.x, en.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break      
            if portMade == True:
                break
            for sn in stbNodes_list:
                if round(sn.x,0) == roundedX and round(sn.y,0) == roundedYMinus:
                    # print("---4/2---")
                    ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=stbNodeIdMatch[(sn.x, sn.y)], nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                    stbCount += 1
                    portMade = True
                    break
            for rail in motherHorizontalLinks[roundedYMinus]:
                xMax = max(rail.startNode.x, rail.endNode.x)
                xMin = min(rail.startNode.x, rail.endNode.x)
                nodeAlignmentSide = "F"
                if xMax == rail.startNode.y:
                    nodeAlignmentSide = "R"
                else:
                    nodeAlignmentSide = "L"
                if roundedX >= xMin and roundedX <= xMax:
                    # print("---4---")
                    if portMade == True:
                        break
                    for link in horizontalLinks[roundedYMinus]:
                        maxX = max(link.startNode.x, link.endNode.x)
                        minX = min(link.startNode.x, link.endNode.x)
                        if roundedX >= minX and roundedX <= maxX and link.startNode.layer == 0:
                            # print("---4/3---")
                            tmpNode = Node(id = str(nodeId).zfill(6), type = "T", reality = "R", x = roundedX, y = roundedYMinus, layer = 0, waitNode = None, parentLinkId = link.id, relativeDistance = roundedX-link.startNode.x + roundedYMinus-link.startNode.y)
                            nodes_list.append(tmpNode)
                            nodeId += 1
                            stbNodeIdMatch[(stbNode.x, stbNode.y)] = tmpNode.id
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=tmpNode.id, nodeAlignment=nodeAlignmentSide, teachingDone = 0))
                            stbCount += 1
                            portMade = True
                            
                            break
                        elif roundedX == link.startNode.x and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.startNode.id, nodeAllignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                        elif roundedX == link.endNode.x and link.startNode.layer == 0:
                            ports_list.append(Port(id=portId, type="STB", carrierType="F400", x=roundedX, y=roundedY, nodeId=link.endNode.id, nodeAllignment=nodeAlignmentSide, teachingDone=0))
                            stbCount+=1
                            portMade=True
                            break
                    break
        if portMade == True:
            break
    if portMade == False:
        
        # print("matchingRailNotFound")
        pass
    


print("OVER!!")


직선 Link 변환중...
곡선 Link 변환중...
EQ Port 변환중...
STB Port 변환중...
OVER!!


In [11]:
# Node, Link객체들을 Map 파일로 변환
def ask_saving_map(prompt="저장할 Map 파일 이름: "):
    while True:
        # s = input(prompt).strip().strip('"').strip("'")
        # if not s:
        #     print("⚠️ 비어있습니다. 파일명을 입력하세요.")
        #     continue
        s = "out.map"
        p = Path(s)
        if p.suffix.lower() != ".map":
            p = p.with_suffix(".map")
        # 상대경로면 현재 작업 폴더 기준
        return p

def node_to_map_line(node: Node) -> str:
    return (
    f"NODE/{node.id}/{node.type}/{node.reality}/{int(node.x)}/{int(node.y)}/"
    f"{'' if node.parentLinkId is None else node.parentLinkId}/"
    f"{'' if node.relativeDistance is None else int(node.relativeDistance)}/"
    f"{int(node.layer)}/{'' if node.waitNode is None else node.waitNode}/|1"
)


def link_to_map_line(link: Link) -> str:
    return f"LINK/{link.id}/{link.type}/{link.startNode.id}/{link.endNode.id}/{link.length}///|||||1|0|0"

def port_to_map(port: Port) -> str:
    return f"PORT/{port.id}/{port.type}/{port.carrierType}/{port.x}/{port.y}/0/{port.nodeId}/{port.nodeAlignment}/0/0/0/||1"

def convert_to_map(nodes, links, ports, save_path):
    with open(save_path, "w") as f:
        for node in nodes:
            f.write(node_to_map_line(node) + "\n")
        for link in links:
            f.write(link_to_map_line(link) + "\n")
        for port in ports:
            f.write(port_to_map(port)+"\n")
        # for zone in zones:
        #     f.write(zone_to_map(zone) + "\n")
    print(f"✅ MAP 파일 저장 완료: {save_path}")

convert_to_map(nodes_list, links_list, ports_list, save_path = ask_saving_map())


✅ MAP 파일 저장 완료: out.map
